In [ ]:
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CSV_PATH = "9f56743a.csv"

CHECKPOINT_PATH = "csv_llm_1M_checkpoint.pth"
WEIGHTS_PATH = "csv_llm_1M_weights.pth"

SEED = 42

# Training
batch_size = 16
block_size = 128
training_steps = 2000
eval_interval = 100
eval_iters = 20

learning_rate = 3e-4
weight_decay = 0.01
dropout = 0.1

# Approximately 1M parameters
embedding_dim = 128
num_heads = 4
num_layers = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


# ============================================================
# 2. RANDOM SEED
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. READ CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

print("\n==============================")
print("CSV INFORMATION")
print("==============================")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
for col in df.columns:
    print(" -", col)


# ============================================================
# 4. CONVERT EACH CSV ROW TO TEXT
# ============================================================

def row_to_text(row):
    """
    Convert one CSV row into structured text.
    """

    parts = []

    for column in df.columns:

        value = row[column]

        if pd.notna(value):
            parts.append(
                f"{column}: {value}"
            )

    return " | ".join(parts)


texts = df.apply(
    row_to_text,
    axis=1
).tolist()


# Add explicit row separators
corpus = "\n<ROW>\n".join(texts)

print("\n==============================")
print("CORPUS INFORMATION")
print("==============================")

print("Corpus characters:", len(corpus))

print("\nExample:")
print(corpus[:1000])


# ============================================================
# 5. BUILD CHARACTER TOKENIZER FROM SCRATCH
# ============================================================

characters = sorted(
    list(set(corpus))
)

vocab_size = len(characters)

stoi = {
    ch: i
    for i, ch in enumerate(characters)
}

itos = {
    i: ch
    for ch, i in stoi.items()
}


def encode(text):
    """
    Convert text to token IDs.
    """

    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]


def decode(token_ids):
    """
    Convert token IDs back to text.
    """

    return "".join(
        itos[int(i)]
        for i in token_ids
    )


data = torch.tensor(
    encode(corpus),
    dtype=torch.long
)

print("\nVocabulary size:", vocab_size)
print("Total tokens:", len(data))


# ============================================================
# 6. TRAIN / VALIDATION SPLIT
# ============================================================

split_index = int(
    0.90 * len(data)
)

train_data = data[:split_index]
val_data = data[split_index:]


print("\nTraining tokens:", len(train_data))
print("Validation tokens:", len(val_data))


# ============================================================
# 7. CHECK DATA SIZE
# ============================================================

if len(train_data) <= block_size + 1:
    raise ValueError(
        f"Training corpus is too small for block_size={block_size}. "
        f"Training tokens={len(train_data)}."
    )

if len(val_data) <= block_size + 1:
    print(
        "\nWARNING: Validation set is smaller than block_size."
    )
    print(
        "Validation evaluation will use the training data instead."
    )


# ============================================================
# 8. GET BATCH
# ============================================================

def get_batch(split):

    if split == "train":
        source = train_data

    else:

        if len(val_data) > block_size + 1:
            source = val_data
        else:
            source = train_data

    max_start = (
        len(source)
        - block_size
        - 1
    )

    indices = torch.randint(
        0,
        max_start,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size]
        for i in indices
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1]
        for i in indices
    ])

    return (
        x.to(device),
        y.to(device)
    )


# ============================================================
# 9. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        dropout
    ):

        super().__init__()

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.feed_forward = nn.Sequential(

            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            ),

            nn.Dropout(dropout)
        )


    def forward(self, x):

        T = x.size(1)

        # Causal mask:
        # a token cannot attend to future tokens.
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        normalized = self.ln1(x)

        attention_output, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=causal_mask,
            need_weights=False
        )

        x = x + attention_output

        x = x + self.feed_forward(
            self.ln2(x)
        )

        return x


# ============================================================
# 10. LANGUAGE MODEL
# ============================================================

class TinyCSVLLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size,
        dropout
    ):

        super().__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.block_size = block_size

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Positional embeddings
        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        # Transformer layers
        self.blocks = nn.ModuleList([
            TransformerBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            embedding_dim
        )

        # Output vocabulary prediction
        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )


    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape

        if T > self.block_size:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"block_size {self.block_size}."
            )

        positions = torch.arange(
            0,
            T,
            device=idx.device
        )

        token_embeddings = (
            self.token_embedding(idx)
        )

        position_embeddings = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_embeddings
            + position_embeddings
        )

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.reshape(
                B * T,
                C
            )

            targets_flat = targets.reshape(
                B * T
            )

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss


    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens=300,
        temperature=0.8,
        top_k=20
    ):

        self.eval()

        for _ in range(max_new_tokens):

            # Use only the latest context window
            idx_context = (
                idx[:, -self.block_size:]
            )

            logits, _ = self(
                idx_context
            )

            # Last-token prediction
            logits = logits[:, -1, :]

            # Temperature
            logits = logits / temperature

            # Top-k sampling
            if top_k is not None:

                k = min(
                    top_k,
                    logits.size(-1)
                )

                values, _ = torch.topk(
                    logits,
                    k
                )

                cutoff = values[:, [-1]]

                logits = torch.where(
                    logits < cutoff,
                    torch.full_like(
                        logits,
                        float("-inf")
                    ),
                    logits
                )

            probabilities = F.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            idx = torch.cat(
                [idx, next_token],
                dim=1
            )

        return idx


# ============================================================
# 11. INITIALIZE MODEL FROM SCRATCH
# ============================================================

model = TinyCSVLLM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    block_size=block_size,
    dropout=dropout
).to(device)


# ============================================================
# 12. COUNT PARAMETERS
# ============================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n==============================")
print("MODEL INFORMATION")
print("==============================")

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Model size: "
    f"{total_parameters / 1_000_000:.3f} M parameters"
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

print(
    "Transformer layers:",
    num_layers
)

print(
    "Embedding dimension:",
    embedding_dim
)

print(
    "Attention heads:",
    num_heads
)

print(
    "FFN dimension:",
    embedding_dim * 4
)

print(
    "Context length:",
    block_size
)

print(
    "Vocabulary size:",
    vocab_size
)


# ============================================================
# 13. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)


# ============================================================
# 14. EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in [
        "train",
        "val"
    ]:

        losses = torch.zeros(
            eval_iters
        )

        for k in range(
            eval_iters
        ):

            xb, yb = get_batch(
                split
            )

            _, loss = model(
                xb,
                yb
            )

            losses[k] = (
                loss.item()
            )

        results[split] = (
            losses.mean().item()
        )

    model.train()

    return results


# ============================================================
# 15. TRAIN MODEL
# ============================================================

print("\n==============================")
print("TRAINING")
print("==============================")

model.train()

best_val_loss = float("inf")


for step in range(
    training_steps + 1
):

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    if (
        step % eval_interval == 0
        or step == training_steps
    ):

        losses = estimate_loss()

        train_loss = losses["train"]
        val_loss = losses["val"]

        perplexity = math.exp(
            min(val_loss, 20)
        )

        print(
            f"Step {step:5d} "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Perplexity: {perplexity:.2f}"
        )

        # Save best model
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            torch.save(
                model.state_dict(),
                WEIGHTS_PATH
            )


    # Do not train after final evaluation
    if step == training_steps:
        break


    # --------------------------------------------------------
    # Training batch
    # --------------------------------------------------------

    xb, yb = get_batch(
        "train"
    )

    _, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()


# ============================================================
# 16. SAVE FULL CHECKPOINT
# ============================================================

checkpoint = {

    # Model
    "model_state_dict":
        model.state_dict(),

    # Optimizer
    "optimizer_state_dict":
        optimizer.state_dict(),

    # Tokenizer
    "stoi":
        stoi,

    "itos":
        itos,

    # Architecture
    "vocab_size":
        vocab_size,

    "embedding_dim":
        embedding_dim,

    "num_heads":
        num_heads,

    "num_layers":
        num_layers,

    "block_size":
        block_size,

    "dropout":
        dropout,

    # Model information
    "total_parameters":
        total_parameters,

    # Training information
    "training_steps":
        training_steps,

    "learning_rate":
        learning_rate,

    "best_val_loss":
        best_val_loss,

    # CSV columns
    "csv_columns":
        list(df.columns)
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# Save final model weights separately
torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


print("\n==============================")
print("MODEL SAVED")
print("==============================")

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

print(
    "Weights:",
    WEIGHTS_PATH
)


# ============================================================
# 17. GENERATE TEXT USING TRAINED MODEL
# ============================================================

model.eval()

prompt = "model_type:"

# Make sure prompt characters exist
prompt_ids = encode(prompt)

if len(prompt_ids) == 0:
    raise ValueError(
        "Prompt contains no characters "
        "from the training vocabulary."
    )


input_tensor = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)


generated_tokens = model.generate(
    input_tensor,
    max_new_tokens=300,
    temperature=0.8,
    top_k=20
)


generated_text = decode(
    generated_tokens[0].tolist()
)


print("\n==============================")
print("GENERATED TEXT")
print("==============================")

print(generated_text)


# ============================================================
# 18. RELOAD MODEL FROM SAVED CHECKPOINT
# ============================================================

print("\n==============================")
print("RELOADING SAVED MODEL")
print("==============================")


loaded_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)


loaded_stoi = (
    loaded_checkpoint["stoi"]
)

loaded_itos = (
    loaded_checkpoint["itos"]
)


loaded_model = TinyCSVLLM(

    vocab_size=
        loaded_checkpoint[
            "vocab_size"
        ],

    embedding_dim=
        loaded_checkpoint[
            "embedding_dim"
        ],

    num_heads=
        loaded_checkpoint[
            "num_heads"
        ],

    num_layers=
        loaded_checkpoint[
            "num_layers"
        ],

    block_size=
        loaded_checkpoint[
            "block_size"
        ],

    dropout=
        loaded_checkpoint[
            "dropout"
        ]

).to(device)


loaded_model.load_state_dict(
    loaded_checkpoint[
        "model_state_dict"
    ]
)


loaded_model.eval()


print(
    "Loaded model parameters:",
    f"{loaded_checkpoint['total_parameters']:,}"
)

print(
    "Loaded model size:",
    f"{loaded_checkpoint['total_parameters'] / 1e6:.3f} M"
)

print(
    "Model successfully loaded."
)


# ============================================================
# 19. GENERATION WITH RELOADED MODEL
# ============================================================

def loaded_encode(text):

    return [
        loaded_stoi[ch]
        for ch in text
        if ch in loaded_stoi
    ]


def loaded_decode(ids):

    return "".join(
        loaded_itos[int(i)]
        for i in ids
    )


prompt = "model_type:"

prompt_ids = loaded_encode(
    prompt
)


x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)


generated = loaded_model.generate(
    x,
    max_new_tokens=300,
    temperature=0.8,
    top_k=20
)


result = loaded_decode(
    generated[0].tolist()
)


print("\n==============================")
print("OUTPUT FROM RELOADED MODEL")
print("==============================")

print(result)